# 58. Causal Bayesian Optimization：介入を選び、実験で確かめる

このNotebookでは「工程条件を**観測する**」ことと「条件を実際に**操作する**」ことを区別し、既知DAGを持つ小さな合成工程で、**介入集合と値の提案 → 実験 → 観測登録 → GP再学習**を体験します。

**このNotebookで確認すること**
1. 既知DAGと構造方程式（SCM）から `do` 介入の結果を求める
2. `InterventionGPSurrogate` で集合別のGP事後を確認する
3. 既存BoTorch LogEIから介入集合と候補値を選ぶ
4. `CausalBOStudy.ask()` / `tell()` で本当に実験を追加する
5. 最良値と介入費用の違いを見る

参考：[理論（do演算子・EIの式）](../../docs/theory/28_causal_bayesian_optimization.md) /
[実装ガイド（API内部の対応）](../../docs/guide/28_causal_bayesian_optimization.md) /
[比較Notebook](59_causal_bo_benchmark.ipynb)

**実行の流れ：** Known DAG → do/SCM → 介入データ → 集合別GP → fit/posterior → LogEI/Optimizer → Candidate → SCM実験 → tell → 再学習。実機の安全性や未知の因果構造の推定を扱う例ではありません。


## 1. 観測と介入の違いを体験する

今回は `X → M → Y ← Z` という既知のDAGを使います。XとZが工程で操作できる設定値、Mが中間の状態、Yが最大化したい目的出力です。

構造方程式は `M = 0.25 + 0.75 X`、`Y = -(M-0.8)^2 - 0.5(Z-0.65)^2` とします。式を1から書き直さず、Phase 6で用意した**既存の合成SCM**を再利用します。

`scm.evaluate()` は通常の生成機構を評価し、`scm.evaluate(HardIntervention({"X": ...}))` はXの構造方程式を置き換えて計算します。これは観測データを「同じ値の記録」でフィルタする操作とは違います。


In [ ]:
import torch
import matplotlib.pyplot as plt

from robotorchan.benchmarks import make_causal_synthetic_scenario
from robotorchan.causal import (
    CausalBOStudy,
    CausalObservation,
    HardIntervention,
    propose_causal_intervention,
)

torch.manual_seed(11)
scenario = make_causal_synthetic_scenario()
graph = scenario.graph
print("DAGのノード順序:", graph.topological_order)
print("Yの親:", graph.parents_of("Y"))
print("許された介入集合:", scenario.intervention_sets)


In [ ]:
natural = scenario.scm.evaluate()
do_x = HardIntervention({"X": 0.8})
under_do = scenario.scm.evaluate(do_x)
observed = CausalObservation(natural)                 # 観測だけ
intervened = CausalObservation(under_do, do_x)        # 実行した介入
print("通常の生成値:", natural)
print("do(X=0.8)の結果:", under_do)
print("is_interventional:", observed.is_interventional, intervened.is_interventional)
assert not observed.is_interventional
assert intervened.is_interventional
assert under_do["M"] != natural["M"]


## 2. 実験予算と介入集合を決める

介入集合は `{X}` と `{X,Z}` の2種類に絞ります（`{Z}` も既存APIでは選択可能です）。両者は異なる介入機構です。コストは `{X}:1`、`{X,Z}:3` とします。

`CausalBOStudy` は各集合で**成功した介入データが2件**集まるまではSobol初期候補を返します。初期化時にDAGを学習したり、因果構造を推定したりしません。最初の実験が評価可能なことは、今回のSCMを既知としているからです。


In [ ]:
study = CausalBOStudy(
    graph,
    "Y",
    intervention_sets=(("X",), ("X", "Z")),
    bounds=scenario.bounds,
    intervention_costs={
        frozenset({"X"}): 1.0,
        frozenset({"X", "Z"}): 3.0,
    },
    n_initial_trials=2,
    optimizer="sobol",   # 軽量な候補探索。獲得関数は既存のBoTorch LogEI
    raw_samples=16,
    seed=11,
)
for _ in range(4):
    trial = study.ask()
    measured = scenario.evaluate(trial.intervention)  # 実世界ならここで実験
    completed = study.tell(trial, measured.values)
    print("初期実験", completed.number, dict(completed.intervention.values),
          "Y =", round(completed.observation.values["Y"], 5))
print("成功した介入記録数:", len(study.observations))
assert len(study.observations) == 4


## 3. 介入GPを学習し、Posteriorを見る

`study.fit()` は **介入集合ごとに独立した** `InterventionGPSurrogate` を学習します。これはrobotorchanの既存 `SingleTaskGP.fit()` への委譲です。

`{X}` のGPにとって入力はXだけの1列 `[n,1]` です。 `{X,Z}` のGPは2列 `[n,2]` となります。**観測だけのレコードを介入GPに混ぜず、別の介入集合も混ぜません。**

Posteriorの`mean`は「その集合を介入したら期待される目的値」、`variance`はモデル予測の不確かさです。これらは因果方向の正しさを検証するものではありません。


In [ ]:
surrogates = study.fit()
for gp in surrogates:
    print("介入集合:", gp.intervention_nodes,
          "学習X:", tuple(gp.train_X.shape), "学習Y:", tuple(gp.train_Y.shape))
gp_x = surrogates[0]
with torch.no_grad():
    posterior = gp_x.posterior(torch.tensor([[0.55]], dtype=torch.double))
print("do(X=0.55) の予測平均:", posterior.mean.squeeze().item())
print("予測標準偏差:", posterior.variance.sqrt().squeeze().item())
assert len(surrogates) == 2


## 4. LogEIから「次にどの介入をするか」を提案する

`propose_causal_intervention` は、各GPの`train_Y`から**共通の現在の最良値**を作り、BoTorchの`LogExpectedImprovement`を既存Optimizerで最大化します。

各集合の介入コストは `log(EI) - log(cost)` として順位に反映されます。LogEIを費用で直接割るのではありません。提案スコアは**実測したYではない**ため、その候補の結果を知るには介入実験が必要です。


In [ ]:
proposal = propose_causal_intervention(
    surrogates,
    study.bounds,
    intervention_costs=study.intervention_costs,
    optimizer="sobol",
    raw_samples=16,
)
print("提案do:", dict(proposal.intervention.values))
print("LogEI:", proposal.log_expected_improvement)
print("介入コスト:", proposal.intervention_cost)
print("コスト調整後スコア:", proposal.log_utility)


## 5. Ask-and-Tellで2回の逐次介入実験を回す

冷スタートが終了しているため、この先の`ask()`はGPとLogEIを用いた候補提案に切り替わります。

- **ask**：介入集合と介入値を選択し、Trialを`RUNNING`にする
- **外部実験**：提案されたHard interventionを実施する（ここではSCMで代行）
- **tell**：介入した条件と実測Yを`COMPLETE`として登録する
- **次のask**：新しい記録を使って必要なGPを再学習する

`tell()`しないまま次の`ask()`を呼ぶことはできません。`state=TrialState.FAIL`で失敗も管理できますが、その場合はGPの学習データに入りません。


In [ ]:
for iteration in range(2):
    trial = study.ask()
    assert trial.proposal is not None  # 初期SobolからGPによる提案に移行
    outcome = scenario.evaluate(trial.intervention)  # 真のdo実験
    completed = study.tell(trial, {"Y": outcome.values["Y"]})
    print(
        f"Round {iteration + 1}:",
        "do =", dict(completed.intervention.values),
        "Y =", round(completed.observation.values["Y"], 5),
        "state =", completed.state.name,
    )
print("累積の介入観測件数:", len(study.observations))
assert len(study.observations) == 6


## 6. 最良値とコストの推移を別々に見る

この例は最大化なので、各実験のYを順に`max`で更新した値が最良値です。コストは各集合の固定費用を積算します。安い介入ばかり選んだときに、必ず最良値も高くなるわけではありません。

横軸が**実験回数**の場合と**累積コスト**の場合では、比較の公平性が違います。


In [ ]:
best = -float("inf")
spent = 0.0
best_values = []
costs = []
for trial in study.trials:
    assert trial.observation is not None
    key = frozenset(trial.intervention.values)
    best = max(best, trial.observation.values["Y"])
    spent += study.intervention_costs[key]
    best_values.append(best)
    costs.append(spent)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(range(1, len(best_values) + 1), best_values, marker="o")
axes[0].set(xlabel="Evaluation count", ylabel="Best observed Y", title="Best value")
axes[1].plot(costs, best_values, marker="o")
axes[1].set(xlabel="Cumulative intervention cost", ylabel="Best observed Y",
            title="Best value vs cost")
plt.tight_layout()
plt.show()
print("最終最良値:", best_values[-1], "累積コスト:", costs[-1])


## 7. 実務へ適用する前の注意

ここでは構造が既知の**決定的な合成SCM**なので、観測がどの介入から得られたかを完全に記録できます。実世界では、実験の実施ログ・入力に対する実際の操作・未操作変数の状態・測定雑音を明示してください。

現在のCBOは、既知DAG・集合別GP・連続介入・単一目的・`q=1`・正の固定介入コストが前提です。**観測だけのデータから因果効果を同定したり、因果グラフを発見したりしません。** 保護装置や因果妥当性の検証を置き換えるものではありません。

比較を学びたい方は[59番Notebook](59_causal_bo_benchmark.ipynb)、数式は[Theory 28章](../../docs/theory/28_causal_bayesian_optimization.md)、API内部は[Guide 28章](../../docs/guide/28_causal_bayesian_optimization.md)を参照してください。
